In [23]:
import fitz # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.schema.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(override=True)


True

In [24]:
## Clip Model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 56541.56it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [25]:
### Embedding functions

def embed_image(image_data):
    """
    Embed an image using CLIP.

    Args:
        image_data: Path (str or Path) or PIL.Image in RGB.

    Returns:
        np.ndarray of shape (embed_dim,), L2-normalized, float32.
    """
    # --- Load / validate image ---
    if isinstance(image_data, (str, Path)):
        with Image.open(image_data) as img:
            image = img.convert("RGB")
    elif isinstance(image_data, Image.Image):
        image = image_data.convert("RGB") if image_data.mode != "RGB" else image_data
    else:
        raise TypeError(
            f"embed_image expected str, Path, or PIL.Image, got {type(image_data).__name__}"
        )

    # --- Preprocess ---
    inputs = clip_processor(images=image, return_tensors="pt")
    device = next(clip_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # --- Forward pass ---
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)

        # Some transformers versions / model classes return an output object
        # (BaseModelOutputWithPooling / CLIPOutput) instead of a tensor.
        if torch.is_tensor(outputs):
            features = outputs
        elif hasattr(outputs, "image_embeds"):
            features = outputs.image_embeds
        elif hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output
        else:
            features = outputs[0]

        # --- Normalize to unit vector ---
        features = features / features.norm(dim=-1, keepdim=True)

    # --- To numpy ---
    return features.squeeze().cpu().numpy().astype("float32", copy=False)


def embed_text(text):
    """Embed text using clip."""
    inputs = clip_processor(
        text = text,
        return_tensors = "pt",
        padding = True,
        truncation = True,
        mal_length = 77 # Clip max token length
    )
    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)

        # Handle both cases
        if hasattr(outputs, "norm"):          # already a tensor
            features = outputs
        elif hasattr(outputs, "text_embeds"): # CLIPModel output
            features = outputs.text_embeds
        elif hasattr(outputs, "pooler_output"):  # CLIPTextModelWithPooling
            features = outputs.pooler_output
        else:
            features = outputs[0]             # fallback

        # Normalize
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

In [26]:
# Process PDF
pdf_path = "./XYZ_Q3_2026_Earnings_Report.pdf"
doc = fitz.open(pdf_path)

all_docs = []
all_embeddings = []
image_data_store = {}

splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap = 50)

In [27]:
for i, page in enumerate(doc):

    text = page.get_text()
    if text.strip():
        # Create a temporary document for splitting
        temp_doc = Document(page_content=text, metadata = {"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        # Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)


    # Process images
    # Three important actions:

    # Convert PDF image to PIL format
    # Store as base64 for GPT-4V (Which needs base64 images)
    # Create CLIP embedding for retrieval
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"

            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64

            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata = {"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close() 


In [28]:
all_embeddings

[array([ 1.56396050e-02, -6.18937938e-03, -1.36194443e-02,  4.80214804e-02,
         3.97724733e-02,  2.99199764e-02, -1.05548315e-02,  1.70528710e-01,
         3.09250932e-02, -2.07921695e-02,  1.90129951e-02, -2.29502264e-02,
        -4.70822006e-02, -2.29759398e-03,  3.26602310e-02, -5.28581999e-03,
         2.62505803e-02,  9.83935222e-03, -1.28071932e-02,  1.99634545e-02,
         1.02613289e-04, -6.08092174e-03,  1.05975717e-01, -1.89171247e-02,
        -1.38382823e-03,  2.97547430e-02, -3.42873409e-02,  1.67722255e-02,
        -1.83139611e-02, -4.54656184e-02,  3.59783694e-02, -1.70841254e-02,
        -5.39254919e-02, -2.14843266e-02,  1.60387084e-02,  1.48194879e-02,
        -1.23822521e-02,  4.35850695e-02,  1.56353414e-02,  1.90641060e-02,
        -5.05454354e-02, -1.08114285e-02, -5.26968576e-02, -4.53020167e-03,
        -1.72782578e-02,  4.37572692e-03,  1.20073697e-02, -5.88730490e-03,
        -2.77769100e-02,  3.35491113e-02,  3.47765759e-02,  4.58499562e-05,
         4.0

In [29]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='XYZ CORPORATION\nQ3 2026 Earnings Report\nQuarter Ended September 30, 2026\nOctober 22, 2026 | CONFIDENTIAL\n$181.2M\n$122.1M\n$33.7M\n$2.58\nTotal Revenue\nGross Profit\nNet Income\nGAAP EPS\n+14.3% YoY'),
 Document(metadata={'page': 0, 'type': 'text'}, page_content='Gross Profit\nNet Income\nGAAP EPS\n+14.3% YoY\n+16.8% YoY\n+18.9% YoY\n+41.8% YoY\nXYZ Corporation (NASDAQ: XYZC) is a global technology leader delivering cloud infrastructure, enterprise software, and'),
 Document(metadata={'page': 0, 'type': 'text'}, page_content='professional services to over 12,000 customers across 45 countries. This report presents unaudited financial results for the\nthird quarter ended September 30, 2026.'),
 Document(metadata={'page': 0, 'type': 'text'}, page_content='third quarter ended September 30, 2026.\nCEO COMMENTARY — "Q3 2026 marks another record quarter for XYZ Corporation. Our cloud-first strategy continues to'),
 Document(me

In [30]:
# Create unified FAISS vector store with CLIP embedding
embeddings_array = np.array(all_embeddings)

# Create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None, # We're using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [31]:
llm = init_chat_model(os.getenv("OLLAMA_CHAT_MODEL"))
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [32]:
def retrieve_multimodel(query, k = 5):
    """Unified retrieval using CLIP embeddings for both text and images"""

    # Embed query using CLIP
    query_embedding = embed_text(query)

    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )

    return results

In [33]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for Gemma4"""
    content = []

    # Add query
    content.append(
        {
            "type": "text",
            "text": f"Question: {query}\n\nContext:\n"
        }
    )

    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append(
            {
                "type": "text",
                "text": f"Text excerpts:\n{text_context}\n"
            }
        )

    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append(
                {
                    "type": "text",
                    "text": f"\n[Image from page {doc.metadata['page']}]:\n"
                }
            )
            content.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{image_data_store[image_id]}"} 
                }
            )

    # Add instruction
    content.append(
        {
            "type": "text",
            "text": "\n\nPlease answer the question based on the provided text and image"
        }
    )

    return HumanMessage(content=content)

In [34]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodel RAG"""

    # Retrieve relevent documents
    context_docs = retrieve_multimodel(query, k = 5)

    # Create multimodel message
    message = create_multimodal_message(query, context_docs)

    # Get response from LLm
    response = llm.invoke([message])

    # Print retrieved context info
    print(f"\nRetrived {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f" - Text from page {page}: {preview}")
        else:
            print(f" - Image from page {page}")

    print("\n")

    return response.content

In [35]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 2 show about revenue trends?",
        "Summarize the main finding from the document",
        "What visual elements are present in the document?"
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-"*150)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("="*150)


Query: What does the chart on page 2 show about revenue trends?
------------------------------------------------------------------------------------------------------------------------------------------------------

Retrived 5 documents:
 - Text from page 4: 3,820
4,140
+8.4%
Revenue / Employee ($K)
$148
$174
+17.6%
 - Text from page 2: of $181.2M represents a $17.8M increase versus the prior quarter and $22.5M versus Q3 2025.
 - Text from page 4: 112%
118%
+600 bps
Annual Recurring Revenue
$468M
$536M
+14.5%
R&D Spend (% of Rev)
14.1%
13.8%
-30 ...
 - Text from page 5: This report contains forward-looking statements based on management's current expectations. Actual r...
 - Text from page 3: 26.3%, and net margin of 18.6% each represent company records, reflecting operational leverage and i...


Answer: Based on the provided text excerpt from Page 2, the chart shows that revenue is trending upward. Specifically, the revenue figure of **$181.2M** represents:

*   A **$17.8M increase**